In [ ]:
"""
Pull DataSF data for the rent-estimating tool and write it into an Excel workbook.

Variables covered (the ones sourced from DataSF):
  #2  Bedrooms, bathrooms & square footage -> Assessor Historical Secured Property Tax Rolls (wv5m-vpq2)
  #4  Unit condition & renovations         -> Building Permits (i98e-djp9)
  #13 Neighborhood crime reports           -> Police Department Incident Reports: 2018 to Present (wg3w-h783)

Usage:
    pip install pandas openpyxl requests
    python datasf_rent_data.py                    # writes "Rent data.xlsx" next to this script
    python datasf_rent_data.py "C:/path/Rent data.xlsx"

If the workbook already exists, only the DataSF sheets are replaced; other sheets are left alone.
Close the workbook in Excel before running.

Optional: set a free Socrata app token in the SOCRATA_APP_TOKEN environment variable to avoid
throttling (https://data.sf.gov/profile/edit/developer_settings).
"""

import os
import sys
import time
from datetime import date, datetime

import pandas as pd
import requests

BASE_URL = "https://data.sf.gov/resource/{}.json"
ASSESSOR = "wv5m-vpq2"
PERMITS = "i98e-djp9"
INCIDENTS = "wg3w-h783"

PAGE_SIZE = 50_000
PERMIT_YEARS_BACK = 10
MIN_UNITS_FOR_RATE = 1_000
MIN_PERMIT_COST = 5_000  # drops water-heater / minor-repair permits that say little about condition

RESIDENTIAL_USES = ("Single Family Residential", "Multi-Family Residential")
PERMIT_USES = ("apartments", "1 family dwelling", "2 family dwelling")
# 1/2 = new construction, 3 = additions/alterations/repairs, 8 = over-the-counter alterations
PERMIT_TYPES = ("1", "2", "3", "8")

VIOLENT = ["Homicide", "Rape", "Robbery", "Assault", "Sex Offense"]
PROPERTY = ["Burglary", "Larceny Theft", "Motor Vehicle Theft", "Arson", "Malicious Mischief", "Vandalism"]

SESSION = requests.Session()
if os.environ.get("SOCRATA_APP_TOKEN"):
    SESSION.headers["X-App-Token"] = os.environ["SOCRATA_APP_TOKEN"]


def soql(dataset, query):
    """Run one SoQL query, retrying on timeouts / throttling."""
    for attempt in range(5):
        try:
            r = SESSION.get(BASE_URL.format(dataset), params={"$query": query}, timeout=300)
            if r.status_code == 200:
                return r.json()
            if r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"{dataset}: {r.status_code} {r.text[:500]}")
        except (requests.Timeout, requests.ConnectionError):
            pass
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"{dataset}: gave up after 5 attempts")


def fetch_all(dataset, select, where, order):
    rows = []
    while True:
        page = soql(dataset, f"select {select} where {where} order by {order} "
                             f"limit {PAGE_SIZE} offset {len(rows)}")
        rows.extend(page)
        print(f"    {len(rows):,} rows", flush=True)
        if len(page) < PAGE_SIZE:
            return pd.DataFrame(rows)


def in_list(values):
    return "(" + ", ".join(f"'{v}'" for v in values) + ")"


def to_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df.get(c), errors="coerce")


def to_date(df, cols):
    for c in cols:
        df[c] = pd.to_datetime(df.get(c), errors="coerce").dt.normalize()


# --------------------------------------------------------------------------- #
# 2  Bedrooms, bathrooms & square footage
# --------------------------------------------------------------------------- #
def get_properties():
    roll_year = soql(ASSESSOR, "select max(closed_roll_year) as y")[0]["y"]
    print(f"  Assessor roll year {roll_year}")
    df = fetch_all(
        ASSESSOR,
        "parcel_number, block, lot, property_location, use_definition, "
        "property_class_code_definition, year_property_built, number_of_bedrooms, "
        "number_of_bathrooms, number_of_rooms, number_of_stories, number_of_units, "
        "property_area, basement_area, lot_area, construction_type, zoning_code, "
        "analysis_neighborhood, assessor_neighborhood, supervisor_district, "
        "current_sales_date, the_geom",
        f"closed_roll_year = '{roll_year}' and use_definition in {in_list(RESIDENTIAL_USES)}",
        "row_id",
    )
    to_numeric(df, ["year_property_built", "number_of_bedrooms", "number_of_bathrooms",
                    "number_of_rooms", "number_of_stories", "number_of_units",
                    "property_area", "basement_area", "lot_area"])
    to_date(df, ["current_sales_date"])
    # property_location is fixed-width, e.g. "0000 0704 NORTH POINT         ST0000" (trailing 4 = unit)
    loc = df.pop("property_location").str.split().str.join(" ").str.extract(r"^\w+ (\w+) (.*?)(\w{4})$")
    df.insert(3, "address", (loc[0].str.lstrip("0") + " " + loc[1].str.strip()).str.strip())
    df.insert(4, "unit", loc[2].str.lstrip("0").replace("", None))
    geom = df.pop("the_geom")
    df["longitude"] = geom.map(lambda g: g["coordinates"][0] if isinstance(g, dict) else None)
    df["latitude"] = geom.map(lambda g: g["coordinates"][1] if isinstance(g, dict) else None)

    units = df["number_of_units"].where(df["number_of_units"] > 0)
    df["sqft_per_unit"] = (df["property_area"].where(df["property_area"] > 0) / units).round(0)
    df.insert(0, "roll_year", int(roll_year))
    return df.rename(columns={
        "property_class_code_definition": "property_class",
        "year_property_built": "year_built",
        "number_of_bedrooms": "bedrooms",
        "number_of_bathrooms": "bathrooms",
        "number_of_rooms": "rooms",
        "number_of_stories": "stories",
        "number_of_units": "units",
        "property_area": "building_sqft",
        "basement_area": "basement_sqft",
        "lot_area": "lot_sqft",
        "current_sales_date": "last_sale_date",
    })


# --------------------------------------------------------------------------- #
# 4  Unit condition & renovations
# --------------------------------------------------------------------------- #
def get_permits():
    today = date.today()
    since = today.replace(year=today.year - PERMIT_YEARS_BACK, day=1)
    df = fetch_all(
        PERMITS,
        "permit_number, permit_type_definition, permit_creation_date, block, lot, "
        "street_number, street_name, street_suffix, unit, description, status, issued_date, "
        "completed_date, estimated_cost, revised_cost, existing_use, proposed_use, "
        "existing_units, proposed_units, adu, neighborhoods_analysis_boundaries, zipcode",
        f"permit_creation_date >= '{since}' and permit_type in {in_list(PERMIT_TYPES)} "
        f"and status in ('complete', 'issued') "
        f"and (existing_use in {in_list(PERMIT_USES)} or proposed_use in {in_list(PERMIT_USES)})",
        "record_id",
    )
    to_numeric(df, ["estimated_cost", "revised_cost", "existing_units", "proposed_units"])
    to_date(df, ["permit_creation_date", "issued_date", "completed_date"])
    df["cost"] = df["revised_cost"].fillna(df["estimated_cost"])
    df = df[df["cost"] >= MIN_PERMIT_COST].copy()
    df["parcel_number"] = df["block"].fillna("") + df["lot"].fillna("")
    # a permit is listed once per street address it covers; keep one row per permit and parcel
    df = df.drop_duplicates(["permit_number", "parcel_number"]).copy()

    df["address"] = (df[["street_number", "street_name", "street_suffix"]]
                     .fillna("").agg(" ".join, axis=1).str.split().str.join(" "))
    desc = df["description"].fillna("").str.lower()
    df["kitchen_work"] = desc.str.contains("kitchen")
    df["bath_work"] = desc.str.contains("bath")
    df = df.rename(columns={"neighborhoods_analysis_boundaries": "analysis_neighborhood",
                            "permit_type_definition": "permit_type",
                            "permit_creation_date": "created_date"})

    detail = df[["permit_number", "parcel_number", "block", "lot", "address", "unit",
                 "analysis_neighborhood", "zipcode", "permit_type", "status", "created_date",
                 "issued_date", "completed_date", "cost", "estimated_cost", "revised_cost",
                 "existing_use", "proposed_use", "existing_units", "proposed_units", "adu",
                 "kitchen_work", "bath_work", "description"]]
    detail = detail.sort_values(["parcel_number", "created_date"], ascending=[True, False])

    by_parcel = df.groupby("parcel_number", as_index=False).agg(
        block=("block", "first"),
        lot=("lot", "first"),
        address=("address", "first"),
        analysis_neighborhood=("analysis_neighborhood", "first"),
        permit_count=("permit_number", "nunique"),
        total_cost=("cost", "sum"),
        largest_permit_cost=("cost", "max"),
        last_permit_issued=("issued_date", "max"),
        last_permit_completed=("completed_date", "max"),
    )
    for flag, name in (("kitchen_work", "last_kitchen_permit"), ("bath_work", "last_bath_permit")):
        last = df[df[flag]].groupby("parcel_number")["issued_date"].max().rename(name)
        by_parcel = by_parcel.merge(last, on="parcel_number", how="left")
    return detail, by_parcel, since


# --------------------------------------------------------------------------- #
# 13  Neighborhood crime reports
# --------------------------------------------------------------------------- #
def crime_counts(start, end):
    rows = soql(
        INCIDENTS,
        "select analysis_neighborhood, incident_category, count(distinct incident_number) as n "
        f"where incident_datetime >= '{start}' and incident_datetime < '{end}' "
        "and analysis_neighborhood is not null and incident_category is not null "
        "group by analysis_neighborhood, incident_category limit 50000",
    )
    df = pd.DataFrame(rows)
    df["n"] = pd.to_numeric(df["n"])
    return df.pivot_table(index="analysis_neighborhood", columns="incident_category",
                          values="n", aggfunc="sum", fill_value=0)


def get_crime(properties):
    # last 12 complete months vs. the 12 before that
    end = date.today().replace(day=1)
    mid = end.replace(year=end.year - 1)
    start = end.replace(year=end.year - 2)
    cur, prior = crime_counts(mid, end), crime_counts(start, mid)

    def rollup(p, cats):
        return p.reindex(columns=cats, fill_value=0).sum(axis=1)

    out = pd.DataFrame({
        "total_incidents": cur.sum(axis=1),
        "violent_incidents": rollup(cur, VIOLENT),
        "property_incidents": rollup(cur, PROPERTY),
        "total_incidents_prior_12mo": prior.sum(axis=1),
    })
    out["total_pct_change"] = (out["total_incidents"] / out["total_incidents_prior_12mo"] - 1).round(3)
    units = properties.groupby("analysis_neighborhood")["units"].sum().rename("residential_units")
    units = units.where(units >= MIN_UNITS_FOR_RATE)  # parks etc. have too few homes for a rate
    out = out.join(units)
    for c in ("total", "violent", "property"):
        out[f"{c}_per_1000_units"] = (out[f"{c}_incidents"] / out["residential_units"] * 1000).round(1)
    out = out.join(cur[cur.sum().sort_values(ascending=False).index])  # one column per category
    out = out.sort_values("total_incidents", ascending=False).reset_index()
    out.insert(1, "period_end", pd.Timestamp(end) - pd.Timedelta(days=1))
    out.insert(1, "period_start", pd.Timestamp(mid))
    return out


# --------------------------------------------------------------------------- #
def write_workbook(path, sheets):
    mode = "a" if os.path.exists(path) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}
    with pd.ExcelWriter(path, engine="openpyxl", mode=mode,
                        datetime_format="yyyy-mm-dd", **kwargs) as writer:
        for name, df in sheets.items():
            print(f"  writing {name} ({len(df):,} rows)", flush=True)
            df.to_excel(writer, sheet_name=name, index=False)
            ws = writer.sheets[name]
            ws.freeze_panes = "A2"
            if len(df):
                ws.auto_filter.ref = ws.dimensions
            for i, col in enumerate(df.columns):
                width = max(len(str(col)), int(df[col].astype(str).str.len().quantile(0.9)) if len(df) else 0)
                ws.column_dimensions[ws.cell(row=1, column=i + 1).column_letter].width = min(width + 2, 60)


def main():
    default = os.path.join(os.path.dirname(os.path.abspath(__file__)), "Rent data.xlsx")
    path = sys.argv[1] if len(sys.argv) > 1 else default

    print("Property characteristics (assessor roll)...")
    properties = get_properties()
    print("Building permits...")
    permits, permits_by_parcel, permits_since = get_permits()
    print("Crime by neighborhood...")
    crime = get_crime(properties)

    readme = pd.DataFrame([
        ("Pulled", datetime.now().strftime("%Y-%m-%d %H:%M")),
        ("Source", "https://data.sf.gov (Socrata API)"),
        ("", ""),
        ("Property_Characteristics",
         f"Variable 2. Assessor secured roll (dataset {ASSESSOR}), roll year {properties['roll_year'].iloc[0]}, "
         "single- and multi-family residential parcels. One row per parcel."),
        ("", "For apartment buildings, bedrooms/bathrooms/sqft are whole-building totals (and bedrooms is "
             "often 0); use sqft_per_unit and units. Condos are one parcel per unit, so their values are per unit."),
        ("Permits_Detail",
         f"Variable 4. Building permits (dataset {PERMITS}) created since {permits_since}, status complete or "
         f"issued, residential use, new construction or alterations, cost >= ${MIN_PERMIT_COST:,}. "
         "cost = revised_cost, or estimated_cost when no revised cost."),
        ("Permits_By_Parcel",
         "Variable 4. Permits_Detail summarized per parcel: count, total cost and most recent "
         "renovation / kitchen / bath permit dates. kitchen/bath flags are keyword matches on the description."),
        ("Crime_By_Neighborhood",
         f"Variable 13. Police incident reports (dataset {INCIDENTS}), distinct incidents in the last 12 "
         "complete months vs. the prior 12, by analysis neighborhood."),
        ("", f"violent = {', '.join(VIOLENT)}. property = {', '.join(PROPERTY)}."),
        ("", "per_1000_units divides by residential units on the assessor roll, not population; it "
             "overstates crime where there are many visitors relative to homes (e.g. South of Market). "
             f"Blank where a neighborhood has under {MIN_UNITS_FOR_RATE:,} units (parks, Treasure Island)."),
        ("Joining the sheets",
         "parcel_number (block + lot) links Property_Characteristics to the permit sheets; "
         "analysis_neighborhood links all of them to Crime_By_Neighborhood."),
    ], columns=["Item", "Notes"])

    print(f"Writing {path}")
    write_workbook(path, {
        "README": readme,
        "Property_Characteristics": properties,
        "Permits_Detail": permits,
        "Permits_By_Parcel": permits_by_parcel,
        "Crime_By_Neighborhood": crime,
    })
    print("Done.")


if __name__ == "__main__":
    main()